# Skin Cancer Detection (Medical Image Classification)Primary model: YOLO26m-cls (with practical checkpoint fallback).Dataset source: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000

## Environment Setup

In [ ]:
import importlibimport subprocessimport sysdef ensure_package(import_name: str, pip_name: str | None = None) -> None:    pkg = pip_name or import_name    try:        importlib.import_module(import_name)    except Exception:        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])ensure_package('kagglehub')ensure_package('numpy')ensure_package('pandas')ensure_package('PIL', 'Pillow')ensure_package('matplotlib')ensure_package('sklearn', 'scikit-learn')ensure_package('ultralytics')ensure_package('yaml', 'pyyaml')print('Dependencies are ready.')

## Imports and Configuration

In [ ]:
import jsonimport osimport randomimport shutilfrom pathlib import Pathimport matplotlib.pyplot as pltimport numpy as npimport pandas as pdfrom PIL import Imagefrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, top_k_accuracy_scoreSEED = 42random.seed(SEED)np.random.seed(SEED)PROJECT_DIR = Path.cwd()WORK_DIR = PROJECT_DIR / 'working' / 'ham10000'PREP_DIR = WORK_DIR / 'prepared_cls'ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'for d in [WORK_DIR, PREP_DIR, ARTIFACTS_DIR]:    d.mkdir(parents=True, exist_ok=True)IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}MAX_TRAIN_PER_CLASS = 500MAX_VAL_PER_CLASS = 200MAX_EVAL_SAMPLES = 1400print(f'Project dir: {PROJECT_DIR}')

## Real Dataset Download

In [ ]:
import kagglehubif not os.getenv('KAGGLE_USERNAME') or not os.getenv('KAGGLE_KEY'):    raise EnvironmentError('Missing Kaggle credentials. Set KAGGLE_USERNAME and KAGGLE_KEY.')dataset_root = Path(kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000'))print(f'Dataset root: {dataset_root}')

## Verify Metadata, Files, Labels, and Build Splits

In [ ]:
meta_path = dataset_root / 'HAM10000_metadata.csv'if not meta_path.exists():    raise FileNotFoundError(f'Missing metadata file: {meta_path}')df_meta = pd.read_csv(meta_path)required_cols = {'image_id', 'dx'}if not required_cols.issubset(set(df_meta.columns)):    raise RuntimeError(f'Metadata missing required columns: {required_cols}')image_dir_candidates = [    dataset_root / 'HAM10000_images',    dataset_root / 'HAM10000_images_part_1',    dataset_root / 'HAM10000_images_part_2']available_dirs = [d for d in image_dir_candidates if d.exists() and d.is_dir()]if len(available_dirs) == 0:    raise RuntimeError('No HAM10000 image directories found.')img_index = {}for d in available_dirs:    for p in d.iterdir():        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:            img_index[p.stem] = pdf_meta['image_path'] = df_meta['image_id'].map(lambda x: str(img_index.get(str(x), '')))missing_paths = (df_meta['image_path'] == '').sum()if missing_paths > 0:    raise RuntimeError(f'Metadata rows without matching image files: {missing_paths}')df_meta = df_meta.rename(columns={'dx': 'label'})class_list = sorted(df_meta['label'].unique().tolist())if len(class_list) < 2:    raise RuntimeError('Expected multiple classes for classification.')# Patient-level split to reduce leakagegroup_col = 'lesion_id' if 'lesion_id' in df_meta.columns else 'image_id'train_parts = []val_parts = []for cls in class_list:    cls_df = df_meta[df_meta['label'] == cls].copy()    group_vals = cls_df[group_col].dropna().astype(str).unique().tolist()    rng = random.Random(SEED)    rng.shuffle(group_vals)    split_idx = max(1, int(0.8 * len(group_vals)))    train_groups = set(group_vals[:split_idx])    train_cls = cls_df[cls_df[group_col].astype(str).isin(train_groups)].copy()    val_cls = cls_df[~cls_df[group_col].astype(str).isin(train_groups)].copy()    if len(val_cls) == 0 and len(train_cls) > 1:        val_cls = train_cls.sample(n=1, random_state=SEED)        train_cls = train_cls.drop(val_cls.index)    train_parts.append(train_cls.assign(split='train'))    val_parts.append(val_cls.assign(split='val'))train_df = pd.concat(train_parts, ignore_index=True)val_df = pd.concat(val_parts, ignore_index=True)df = pd.concat([train_df, val_df], ignore_index=True)if len(df) == 0:    raise RuntimeError('No rows after split creation.')print(f'Total rows: {len(df_meta)}')print(f'Classes: {class_list}')print(f'Train rows: {len(train_df)}, Val rows: {len(val_df)}')print(train_df['label'].value_counts().sort_index())print(val_df['label'].value_counts().sort_index())df.head()

In [ ]:
# Sample image integrity checksample_paths = df['image_path'].sample(n=min(500, len(df)), random_state=SEED).tolist()corrupt = 0for p in sample_paths:    try:        with Image.open(p) as img:            img.verify()    except Exception:        corrupt += 1if corrupt > 0:    raise RuntimeError(f'Corrupted image files detected in sample check: {corrupt}')print('Sampled integrity check passed.')

## Prepare YOLO Classification Directory

In [ ]:
for split_name in ['train', 'val']:    for cls in class_list:        (PREP_DIR / split_name / cls).mkdir(parents=True, exist_ok=True)def cap_by_class(source_df: pd.DataFrame, limit_per_class: int) -> pd.DataFrame:    parts = []    for cls in class_list:        chunk = source_df[source_df['label'] == cls].copy()        if len(chunk) > limit_per_class:            chunk = chunk.sample(n=limit_per_class, random_state=SEED)        parts.append(chunk)    return pd.concat(parts, ignore_index=True)train_small = cap_by_class(train_df, MAX_TRAIN_PER_CLASS)val_small = cap_by_class(val_df, MAX_VAL_PER_CLASS)for _, row in train_small.iterrows():    src = Path(row['image_path'])    dst = PREP_DIR / 'train' / row['label'] / src.name    if not dst.exists():        shutil.copy2(src, dst)for _, row in val_small.iterrows():    src = Path(row['image_path'])    dst = PREP_DIR / 'val' / row['label'] / src.name    if not dst.exists():        shutil.copy2(src, dst)print(f'Prepared train rows: {len(train_small)}')print(f'Prepared val rows: {len(val_small)}')

In [ ]:
def count_images(root: Path) -> int:    return sum(1 for p in root.rglob('*') if p.suffix.lower() in IMAGE_EXTS)n_train = count_images(PREP_DIR / 'train')n_val = count_images(PREP_DIR / 'val')if n_train == 0 or n_val == 0:    raise RuntimeError('Prepared train/val image counts are zero.')print(f'Train image files: {n_train}')print(f'Val image files: {n_val}')

## Real Data Visualization

In [ ]:
viz_df = train_small.sample(n=min(9, len(train_small)), random_state=SEED).reset_index(drop=True)fig, axes = plt.subplots(3, 3, figsize=(11, 10))for i in range(len(viz_df)):    row = viz_df.iloc[i]    img = Image.open(row['image_path']).convert('RGB')    axes.flatten()[i].imshow(img)    axes.flatten()[i].set_title(row['label'])    axes.flatten()[i].axis('off')for j in range(len(viz_df), 9):    axes.flatten()[j].axis('off')plt.tight_layout()sample_grid = ARTIFACTS_DIR / 'sample_grid.png'plt.savefig(sample_grid, dpi=140)plt.show()

## Train YOLO26m-cls

In [ ]:
from ultralytics import YOLOweights_candidates = ['yolo26m-cls.pt', 'yolo11m-cls.pt', 'yolov8m-cls.pt']selected_weights = Nonemodel = Nonefor w in weights_candidates:    try:        model = YOLO(w)        selected_weights = w        break    except Exception:        continueif selected_weights is None:    raise RuntimeError('Could not load a YOLO classification checkpoint.')print(f'Selected checkpoint: {selected_weights}')train_result = model.train(    data=str(PREP_DIR),    epochs=2,    imgsz=160,    batch=64,    project=str(ARTIFACTS_DIR / 'yolo_runs'),    name='skin_cancer_cls',    seed=SEED,    verbose=False)best_model_path = Path(model.trainer.best)print(f'Best model path: {best_model_path}')

## Real Evaluation

In [ ]:
best_model = YOLO(str(best_model_path))label_to_idx = {name: i for i, name in enumerate(sorted(class_list))}idx_to_label = {v: k for k, v in label_to_idx.items()}eval_df = val_small.copy().reset_index(drop=True)if len(eval_df) > MAX_EVAL_SAMPLES:    eval_df = eval_df.sample(n=MAX_EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)y_true = []y_pred = []y_prob_rows = []for _, row in eval_df.iterrows():    pred = best_model.predict(source=row['image_path'], verbose=False)[0]    probs = pred.probs.data.cpu().numpy()    y_true.append(label_to_idx[row['label']])    y_pred.append(int(np.argmax(probs)))    y_prob_rows.append(probs)y_prob = np.vstack(y_prob_rows)acc = accuracy_score(y_true, y_pred)macro_f1 = f1_score(y_true, y_pred, average='macro')top3 = top_k_accuracy_score(y_true, y_prob, k=3, labels=list(range(len(class_list))))cm = confusion_matrix(y_true, y_pred)print(f'Eval samples: {len(eval_df)}')print(f'Accuracy: {acc:.4f}')print(f'Macro F1: {macro_f1:.4f}')print(f'Top-3 accuracy: {top3:.4f}')print(classification_report(y_true, y_pred, labels=list(range(len(class_list))), target_names=sorted(class_list), zero_division=0))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))ax.imshow(cm, cmap='Blues')ax.set_title('Confusion Matrix (All Classes)')ax.set_xlabel('Predicted')ax.set_ylabel('True')ticks = list(range(len(class_list)))ax.set_xticks(ticks)ax.set_yticks(ticks)ax.set_xticklabels(sorted(class_list), rotation=45, ha='right')ax.set_yticklabels(sorted(class_list))plt.tight_layout()cm_path = ARTIFACTS_DIR / 'confusion_matrix.png'plt.savefig(cm_path, dpi=140)plt.show()

## Honest Qualitative Analysis

In [ ]:
pred_df = eval_df.copy()pred_df['true_label'] = [idx_to_label[i] for i in y_true]pred_df['pred_label'] = [idx_to_label[i] for i in y_pred]pred_df['correct'] = (pred_df['true_label'] == pred_df['pred_label']).astype(int)hard = pred_df[pred_df['correct'] == 0].head(9)if len(hard) == 0:    hard = pred_df.sample(n=min(9, len(pred_df)), random_state=SEED)fig, axes = plt.subplots(3, 3, figsize=(11, 10))for i in range(len(hard)):    row = hard.iloc[i]    img = Image.open(row['image_path']).convert('RGB')    axes.flatten()[i].imshow(img)    axes.flatten()[i].set_title(f"t={row['true_label']} p={row['pred_label']}")    axes.flatten()[i].axis('off')for j in range(len(hard), 9):    axes.flatten()[j].axis('off')plt.tight_layout()qual_path = ARTIFACTS_DIR / 'qualitative_cases.png'plt.savefig(qual_path, dpi=140)plt.show()

## Save Real Outputs

In [ ]:
pred_csv = ARTIFACTS_DIR / 'eval_predictions.csv'pred_df.to_csv(pred_csv, index=False)metrics = {    'dataset': 'kmader/skin-cancer-mnist-ham10000',    'dataset_url': 'https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000',    'num_classes_total': int(len(class_list)),    'num_rows_total': int(len(df_meta)),    'num_train_rows_original': int(len(train_df)),    'num_val_rows_original': int(len(val_df)),    'num_train_rows_prepared': int(len(train_small)),    'num_val_rows_prepared': int(len(val_small)),    'selected_weights': selected_weights,    'best_model_path': str(best_model_path),    'eval_samples': int(len(eval_df)),    'accuracy': float(acc),    'macro_f1': float(macro_f1),    'top3_accuracy': float(top3),    'max_train_per_class': int(MAX_TRAIN_PER_CLASS),    'max_val_per_class': int(MAX_VAL_PER_CLASS),    'max_eval_samples': int(MAX_EVAL_SAMPLES)}metrics_path = ARTIFACTS_DIR / 'metrics.json'with open(metrics_path, 'w', encoding='utf-8') as f:    json.dump(metrics, f, indent=2)manifest = {    'dataset_root': str(dataset_root),    'prepared_data_root': str(PREP_DIR),    'metrics_json': str(metrics_path),    'predictions_csv': str(pred_csv),    'sample_grid_png': str(sample_grid),    'confusion_matrix_png': str(cm_path),    'qualitative_png': str(qual_path)}manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'with open(manifest_path, 'w', encoding='utf-8') as f:    json.dump(manifest, f, indent=2)print('Saved outputs:')print(metrics_path)print(pred_csv)print(sample_grid)print(cm_path)print(qual_path)print(manifest_path)

## Limitations- The notebook uses practical row caps and short epochs for local execution constraints.- For stronger clinical-grade robustness, train longer, calibrate probabilities, and validate on external cohorts.- This is an educational workflow and not a medical diagnostic system.